# BirdCLEF 2026 Data Setup to Google Drive

This notebook downloads the BirdCLEF 2026 competition data into Colab runtime storage, preserves the original Kaggle archive in Google Drive, extracts the visible dataset, and verifies the final Drive manifest.

Expected persistent Drive root:

`MyDrive/birdclef-2026/`


## Workflow

1. Mount Google Drive.
2. Install Kaggle in Colab if needed.
3. Configure Kaggle auth (`KAGGLE_API_TOKEN` preferred; `kaggle.json` fallback).
4. Download `birdclef-2026` into `/content/birdclef-2026-download`.
5. Copy the raw Kaggle archive into Drive under `raw/kaggle-download/`.
6. Extract the archive in Colab runtime.
7. Copy the extracted visible files into Drive under `data/`.
8. Verify the Drive manifest.


In [16]:
# Step 1: Mount Google Drive so the dataset can be stored persistently.
print('[STEP 1/8] Mounting Google Drive...')

from google.colab import drive

drive.mount('/content/drive')
print('[OK] Google Drive mounted at /content/drive')


[STEP 1/8] Mounting Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[OK] Google Drive mounted at /content/drive


In [17]:
# Step 2: Install Kaggle inside the Colab runtime.
print('[STEP 2/8] Installing Kaggle CLI in Colab...')
import subprocess
import sys

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'kaggle'], check=True)
print('[OK] Kaggle CLI installation finished.')


[STEP 2/8] Installing Kaggle CLI in Colab...
[OK] Kaggle CLI installation finished.


In [18]:
# Step 3 prep: Configure paths and helper utilities used throughout the notebook.
# The helper functions add explicit logging and progress bars so the notebook shows
# exactly what is happening during copy, extraction, and verification.

from pathlib import Path
import json
import os
import shutil
import subprocess
import zipfile
from tqdm.auto import tqdm

COMPETITION = 'birdclef-2026'
DRIVE_ROOT = Path('/content/drive/MyDrive/birdclef-2026')
RAW_ARCHIVE_DIR = DRIVE_ROOT / 'raw' / 'kaggle-download'
DATA_DIR = DRIVE_ROOT / 'data'
OUTPUTS_DIR = DRIVE_ROOT / 'outputs'
DOCS_DIR = DRIVE_ROOT / 'docs'

RUNTIME_ROOT = Path('/content/birdclef-2026-download')
DOWNLOAD_DIR = RUNTIME_ROOT / 'download'
EXTRACT_DIR = RUNTIME_ROOT / 'extracted'

OUTPUT_DIRS = [
    RAW_ARCHIVE_DIR,
    DATA_DIR,
    OUTPUTS_DIR / 'checkpoints',
    OUTPUTS_DIR / 'oof',
    OUTPUTS_DIR / 'submissions',
    DOCS_DIR,
    DOWNLOAD_DIR,
    EXTRACT_DIR,
]

EXPECTED_ENTRIES = [
    'train_audio',
    'train_soundscapes',
    'train.csv',
    'taxonomy.csv',
    'sample_submission.csv',
    'train_soundscapes_labels.csv',
    'recording_location.txt',
]

REQUIRED_DRIVE_ARTIFACTS = [DATA_DIR / entry for entry in EXPECTED_ENTRIES]
USE_SYSTEM_UNZIP = True
USE_RSYNC_FOR_DIRECTORY_SYNC = True
PRINT_FULL_MANIFEST = False
FAST_MODE = True
ALLOW_UPLOAD_FALLBACK = False
SKIP_ARCHIVE_PREVIEW = True
VERIFY_DEEP_DIRECTORY_STATS = False
SKIP_DOWNLOAD_IF_ARCHIVE_EXISTS = True
SKIP_RAW_ARCHIVE_COPY_IF_EXISTS = True
SKIP_EXTRACT_IF_EXPECTED_EXISTS = True
SKIP_SYNC_IF_ENTRY_EXISTS = True
SKIP_SYNC_IF_FILE_SAME_SIZE = True
STRICT_MANIFEST_SIZE_VALIDATION = True
REDOWNLOAD_ARCHIVE_ON_SIZE_MISMATCH = True
LARGE_FILE_RSYNC_THRESHOLD_BYTES = 512 * 1024 * 1024

print(f'[CONFIG] USE_SYSTEM_UNZIP={USE_SYSTEM_UNZIP}')
print(f'[CONFIG] USE_RSYNC_FOR_DIRECTORY_SYNC={USE_RSYNC_FOR_DIRECTORY_SYNC}')
print(f'[CONFIG] PRINT_FULL_MANIFEST={PRINT_FULL_MANIFEST}')
print(f'[CONFIG] FAST_MODE={FAST_MODE}')
print(f'[CONFIG] ALLOW_UPLOAD_FALLBACK={ALLOW_UPLOAD_FALLBACK}')
print(f'[CONFIG] SKIP_ARCHIVE_PREVIEW={SKIP_ARCHIVE_PREVIEW}')
print(f'[CONFIG] VERIFY_DEEP_DIRECTORY_STATS={VERIFY_DEEP_DIRECTORY_STATS}')
print(f'[CONFIG] SKIP_DOWNLOAD_IF_ARCHIVE_EXISTS={SKIP_DOWNLOAD_IF_ARCHIVE_EXISTS}')
print(f'[CONFIG] SKIP_RAW_ARCHIVE_COPY_IF_EXISTS={SKIP_RAW_ARCHIVE_COPY_IF_EXISTS}')
print(f'[CONFIG] SKIP_EXTRACT_IF_EXPECTED_EXISTS={SKIP_EXTRACT_IF_EXPECTED_EXISTS}')
print(f'[CONFIG] SKIP_SYNC_IF_ENTRY_EXISTS={SKIP_SYNC_IF_ENTRY_EXISTS}')
print(f'[CONFIG] SKIP_SYNC_IF_FILE_SAME_SIZE={SKIP_SYNC_IF_FILE_SAME_SIZE}')
print(f'[CONFIG] STRICT_MANIFEST_SIZE_VALIDATION={STRICT_MANIFEST_SIZE_VALIDATION}')
print(f'[CONFIG] REDOWNLOAD_ARCHIVE_ON_SIZE_MISMATCH={REDOWNLOAD_ARCHIVE_ON_SIZE_MISMATCH}')
print(f'[CONFIG] LARGE_FILE_RSYNC_THRESHOLD_BYTES={LARGE_FILE_RSYNC_THRESHOLD_BYTES}')

def print_banner(message: str) -> None:
    separator = '=' * 80
    print(f'\n{separator}\n{message}\n{separator}')

def format_bytes(num_bytes: int) -> str:
    units = ['B', 'KB', 'MB', 'GB', 'TB']
    value = float(num_bytes)
    for unit in units:
        if value < 1024 or unit == units[-1]:
            return f'{value:.2f} {unit}'
        value /= 1024

def ensure_directories(paths):
    print_banner('[CONFIG] Ensuring runtime and Drive directories exist')
    for path in paths:
        path.mkdir(parents=True, exist_ok=True)
        print(f'[DIR] Ready: {path}')

def command_exists(name: str) -> bool:
    return shutil.which(name) is not None

def run_command(cmd, step_label: str) -> None:
    print(f'[RUN:{step_label}] ' + ' '.join(str(part) for part in cmd))
    subprocess.run(cmd, check=True)
    print(f'[OK:{step_label}] Command completed successfully.')

def copy_file_with_progress(src: Path, dst: Path, chunk_size: int = 64 * 1024 * 1024) -> None:
    dst.parent.mkdir(parents=True, exist_ok=True)
    total = src.stat().st_size
    if command_exists('rsync') and total >= LARGE_FILE_RSYNC_THRESHOLD_BYTES:
        # rsync is typically faster for very large files and provides native progress output.
        cmd = ['rsync', '-a', '--whole-file', '--info=progress2', str(src), str(dst)]
        run_command(cmd, 'RSYNC_FILE')
        return

    print(f'[COPY] File -> {src} -> {dst}')
    print(f'[COPY] Size: {format_bytes(total)}')
    with src.open('rb') as fsrc, dst.open('wb') as fdst, tqdm(
        total=total,
        unit='B',
        unit_scale=True,
        desc=f'Copying {src.name}',
    ) as pbar:
        while True:
            chunk = fsrc.read(chunk_size)
            if not chunk:
                break
            fdst.write(chunk)
            pbar.update(len(chunk))
    shutil.copystat(src, dst)
    print(f'[OK] Copied file to {dst}')

def extract_zip_with_progress(zip_path: Path, destination: Path) -> None:
    print_banner('[EXTRACT] Extracting archive into Colab runtime storage')
    if destination.exists():
        print(f'[CLEANUP] Removing previous extraction directory: {destination}')
        shutil.rmtree(destination)
    destination.mkdir(parents=True, exist_ok=True)

    if USE_SYSTEM_UNZIP and command_exists('unzip'):
        # Native unzip is usually much faster than Python zip extraction on large archives.
        cmd = ['unzip', '-oq', str(zip_path), '-d', str(destination)]
        run_command(cmd, 'UNZIP')
        print(f'[OK] Archive extracted to {destination} using system unzip.')
        return

    print('[INFO] Falling back to Python zip extraction with tqdm progress bar.')

    with zipfile.ZipFile(zip_path) as zf:
        members = zf.infolist()
        print(f'[EXTRACT] Member count: {len(members)}')
        for member in tqdm(members, desc='Extracting archive members', unit='file'):
            zf.extract(member, destination)

    print(f'[OK] Archive extracted to {destination}')

def remove_existing_path(path: Path) -> None:
    if not path.exists():
        return
    print(f'[CLEANUP] Removing existing destination: {path}')
    if path.is_dir():
        shutil.rmtree(path)
    else:
        path.unlink()

def copy_directory_with_progress(src: Path, dst: Path) -> None:
    if USE_RSYNC_FOR_DIRECTORY_SYNC and command_exists('rsync'):
        # rsync is significantly faster and safer for large directory trees.
        dst.mkdir(parents=True, exist_ok=True)
        cmd = ['rsync', '-a', '--whole-file', '--delete', '--info=progress2', f'{src}/', f'{dst}/']
        run_command(cmd, 'RSYNC')
        print(f'[OK] Directory synced with rsync: {src} -> {dst}')
        return

    print('[INFO] rsync unavailable or disabled. Using Python file copy fallback.')
    files = sorted([path for path in src.rglob('*') if path.is_file()])
    total_bytes = sum(path.stat().st_size for path in files)
    print(f'[COPY] Directory -> {src} -> {dst}')
    print(f'[COPY] File count: {len(files)} | Total size: {format_bytes(total_bytes)}')
    dst.mkdir(parents=True, exist_ok=True)

    with tqdm(total=len(files), desc=f'Syncing {src.name}', unit='file') as pbar:
        for file_path in files:
            relative = file_path.relative_to(src)
            target = dst / relative
            target.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(file_path, target)
            pbar.update(1)

    print(f'[OK] Directory synced to {dst}')

def summarize_directory(path: Path, deep: bool = False) -> str:
    if deep:
        files = [entry for entry in path.rglob('*') if entry.is_file()]
        total_bytes = sum(entry.stat().st_size for entry in files)
        return f'{len(files)} files | {format_bytes(total_bytes)}'

    direct_entries = sum(1 for _ in path.iterdir()) if path.exists() and path.is_dir() else 0
    return f'{direct_entries} direct entries | deep stats skipped'

ARCHIVE_FILE_SIZES = {}

def build_archive_file_sizes(zip_path: Path) -> dict:
    with zipfile.ZipFile(zip_path) as zf:
        return {info.filename: info.file_size for info in zf.infolist() if not info.is_dir()}

def expected_sizes_for_entry(entry: str, manifest: dict) -> dict:
    if entry in manifest:
        return {'.': manifest[entry]}
    prefix = f'{entry}/'
    return {
        rel[len(prefix):]: size
        for rel, size in manifest.items()
        if rel.startswith(prefix)
    }

def path_matches_manifest_size(entry: str, path: Path, manifest: dict, show_progress: bool = False):
    expected = expected_sizes_for_entry(entry, manifest)
    if not expected:
        return False, f'No manifest entries found for {entry}.'

    if path.is_file():
        expected_size = expected.get('.')
        if expected_size is None:
            return False, f'Manifest expects a directory, but destination is file: {path}'
        actual_size = path.stat().st_size
        if actual_size != expected_size:
            return False, f'Size mismatch for {path.name}: expected {expected_size}, found {actual_size}'
        return True, f'File matches manifest size: {path.name} ({actual_size} bytes)'

    if not path.is_dir():
        return False, f'Destination path is neither file nor directory: {path}'

    expected_items = sorted((rel, size) for rel, size in expected.items() if rel != '.')
    iterator = expected_items
    if show_progress and len(expected_items) > 2000:
        iterator = tqdm(expected_items, total=len(expected_items), desc=f'Validating {entry}', unit='file')

    for rel, expected_size in iterator:
        target = path / rel
        if not target.exists():
            return False, f'Missing file in destination: {target}'
        if not target.is_file():
            return False, f'Expected file path is not a file: {target}'
        actual_size = target.stat().st_size
        if actual_size != expected_size:
            return False, f'Size mismatch for {target}: expected {expected_size}, found {actual_size}'

    return True, f'{entry}: {len(expected_items)} files match manifest sizes.'

ensure_directories(OUTPUT_DIRS)
print(f'[CONFIG] Competition slug: {COMPETITION}')
print(f'[CONFIG] Drive root: {DRIVE_ROOT}')
print(f'[CONFIG] Runtime root: {RUNTIME_ROOT}')


[CONFIG] USE_SYSTEM_UNZIP=True
[CONFIG] USE_RSYNC_FOR_DIRECTORY_SYNC=True
[CONFIG] PRINT_FULL_MANIFEST=False
[CONFIG] FAST_MODE=True
[CONFIG] ALLOW_UPLOAD_FALLBACK=False
[CONFIG] SKIP_ARCHIVE_PREVIEW=True
[CONFIG] VERIFY_DEEP_DIRECTORY_STATS=False
[CONFIG] SKIP_DOWNLOAD_IF_ARCHIVE_EXISTS=True
[CONFIG] SKIP_RAW_ARCHIVE_COPY_IF_EXISTS=True
[CONFIG] SKIP_EXTRACT_IF_EXPECTED_EXISTS=True
[CONFIG] SKIP_SYNC_IF_ENTRY_EXISTS=True
[CONFIG] SKIP_SYNC_IF_FILE_SAME_SIZE=True
[CONFIG] STRICT_MANIFEST_SIZE_VALIDATION=True
[CONFIG] REDOWNLOAD_ARCHIVE_ON_SIZE_MISMATCH=True
[CONFIG] LARGE_FILE_RSYNC_THRESHOLD_BYTES=536870912

[CONFIG] Ensuring runtime and Drive directories exist
[DIR] Ready: /content/drive/MyDrive/birdclef-2026/raw/kaggle-download
[DIR] Ready: /content/drive/MyDrive/birdclef-2026/data
[DIR] Ready: /content/drive/MyDrive/birdclef-2026/outputs/checkpoints
[DIR] Ready: /content/drive/MyDrive/birdclef-2026/outputs/oof
[DIR] Ready: /content/drive/MyDrive/birdclef-2026/outputs/submissions
[D

## Kaggle Authentication

Preferred order: Drive `secrets` files -> Colab Secrets/env -> upload fallback.\n
If you already uploaded `.env` or `kaggle.json` into `MyDrive/birdclef-2026/secrets`, Step 3 now picks them up automatically.


In [19]:
# Step 3: Configure Kaggle authentication for this Colab session.
# Resolution order:
# 1) Drive secrets folder (MyDrive/birdclef-2026/secrets/.env or kaggle.json)
# 2) Existing environment variable KAGGLE_API_TOKEN
# 3) Colab Secrets (userdata)
# 4) Upload kaggle.json fallback
print_banner('[STEP 3/8] Configuring Kaggle authentication')

from google.colab import files

def parse_env_token(env_path: Path) -> str:
    if not env_path.exists():
        return ''
    for raw_line in env_path.read_text(encoding='utf-8').splitlines():
        line = raw_line.strip()
        if not line or line.startswith('#'):
            continue
        if line.startswith('export '):
            line = line[len('export '):].strip()
        if '=' not in line:
            continue
        key, value = line.split('=', 1)
        if key.strip() == 'KAGGLE_API_TOKEN':
            return value.strip().strip('"').strip("'")
    return ''

def apply_kaggle_json_credentials(kaggle_json_path: Path) -> bool:
    if not kaggle_json_path.exists():
        return False
    with kaggle_json_path.open('r', encoding='utf-8') as fh:
        creds = json.load(fh)
    if 'api_token' in creds:
        os.environ['KAGGLE_API_TOKEN'] = str(creds['api_token']).strip()
        print(f'[OK] Loaded api_token from {kaggle_json_path}.')
        return True
    if 'username' in creds and 'key' in creds:
        os.environ['KAGGLE_USERNAME'] = str(creds['username']).strip()
        os.environ['KAGGLE_KEY'] = str(creds['key']).strip()
        print(f'[OK] Loaded username/key from {kaggle_json_path}.')
        return True
    raise ValueError(f'Unsupported kaggle.json format in {kaggle_json_path}')

kaggle_dir = Path('/root/.kaggle')
kaggle_dir.mkdir(parents=True, exist_ok=True)
os.chmod(kaggle_dir, 0o700)

auth_configured = False
drive_secret_dir_candidates = []
secret_dir_override = os.environ.get('KAGGLE_SECRET_DIR', '').strip()
if secret_dir_override:
    drive_secret_dir_candidates.append(Path(secret_dir_override))
drive_secret_dir_candidates.extend([
    DRIVE_ROOT / 'secrets',
    Path('/content/drive/MyDrive/birdclef-2026/secrets'),
])

seen_secret_dirs = set()
for drive_secret_dir in drive_secret_dir_candidates:
    drive_secret_dir = Path(drive_secret_dir)
    secret_dir_key = str(drive_secret_dir)
    if secret_dir_key in seen_secret_dirs:
        continue
    seen_secret_dirs.add(secret_dir_key)

    drive_env_path = drive_secret_dir / '.env'
    drive_json_path = drive_secret_dir / 'kaggle.json'
    print(f'[INFO] Checking Drive secrets directory: {drive_secret_dir}')
    print(f'[INFO] .env exists={drive_env_path.exists()} | kaggle.json exists={drive_json_path.exists()}')

    if drive_env_path.exists():
        token = parse_env_token(drive_env_path)
        if token:
            os.environ['KAGGLE_API_TOKEN'] = token
            auth_configured = True
            print(f'[OK] Loaded KAGGLE_API_TOKEN from {drive_env_path}.')
            break
        else:
            print(f'[WARN] Found {drive_env_path} but no KAGGLE_API_TOKEN key was parsed.')

    if drive_json_path.exists():
        auth_configured = apply_kaggle_json_credentials(drive_json_path)
        if auth_configured:
            break

if not auth_configured:
    token = os.environ.get('KAGGLE_API_TOKEN', '').strip()
    if token:
        auth_configured = True
        print('[OK] Using KAGGLE_API_TOKEN from environment variable.')

if not auth_configured:
    try:
        from google.colab import userdata
        token = (userdata.get('KAGGLE_API_TOKEN') or '').strip()
        if token:
            os.environ['KAGGLE_API_TOKEN'] = token
            auth_configured = True
            print('[OK] Loaded KAGGLE_API_TOKEN from Colab Secrets.')
    except Exception as exc:
        print(f'[INFO] Colab userdata not available: {exc}')

if not auth_configured:
    if not ALLOW_UPLOAD_FALLBACK:
        raise RuntimeError(
            'No authentication source found. For Cursor Colab extension, avoid upload widgets and either: '
            '1) place .env or kaggle.json in /content/drive/MyDrive/birdclef-2026/secrets, '
            'or 2) set KAGGLE_API_TOKEN in the environment before running Step 3, '
            'or 3) set KAGGLE_SECRET_DIR to a custom Drive path.'
        )

    print('[ACTION] No Drive/env/secrets auth found. Upload kaggle.json now.')
    uploaded = files.upload()
    if 'kaggle.json' not in uploaded:
        raise FileNotFoundError('No authentication source found. Provide Drive secret file, env token, or kaggle.json upload.')
    kaggle_json = kaggle_dir / 'kaggle.json'
    kaggle_json.write_bytes(uploaded['kaggle.json'])
    os.chmod(kaggle_json, 0o600)
    auth_configured = apply_kaggle_json_credentials(kaggle_json)

print('[CHECK] Kaggle CLI version:')
subprocess.run(['kaggle', '--version'], check=True)
if not auth_configured:
    raise RuntimeError('Kaggle authentication was not configured.')
print('[OK] Kaggle authentication configured successfully.')



[STEP 3/8] Configuring Kaggle authentication
[INFO] Checking Drive secrets directory: /content/drive/MyDrive/birdclef-2026/secrets
[INFO] .env exists=True | kaggle.json exists=True
[OK] Loaded KAGGLE_API_TOKEN from /content/drive/MyDrive/birdclef-2026/secrets/.env.
[CHECK] Kaggle CLI version:
[OK] Kaggle authentication configured successfully.


In [20]:
# Step 4: Download the competition archive into temporary Colab runtime storage.
print_banner('[STEP 4/8] Downloading BirdCLEF 2026 competition archive')

archive_candidate = DOWNLOAD_DIR / f'{COMPETITION}.zip'
archive_path = None
if SKIP_DOWNLOAD_IF_ARCHIVE_EXISTS and archive_candidate.exists() and zipfile.is_zipfile(archive_candidate):
    archive_path = archive_candidate
    print(f'[SKIP] Reusing existing runtime archive: {archive_path}')
else:
    cmd = [
        'kaggle', 'competitions', 'download',
        '-c', COMPETITION,
        '-p', str(DOWNLOAD_DIR),
    ]
    print('[RUN] ' + ' '.join(cmd))
    subprocess.run(cmd, check=True)

    archives = sorted(DOWNLOAD_DIR.glob('*.zip'))
    if not archives:
        raise FileNotFoundError('No competition archive was downloaded.')
    archive_path = archive_candidate if archive_candidate.exists() else archives[0]
    print(f'[OK] Downloaded archive: {archive_path}')

if archive_path is None:
    raise RuntimeError('archive_path was not set.')

print(f'[INFO] Using archive: {archive_path}')
print(f'[INFO] Archive size: {format_bytes(archive_path.stat().st_size)}')



[STEP 4/8] Downloading BirdCLEF 2026 competition archive
[SKIP] Reusing existing runtime archive: /content/birdclef-2026-download/download/birdclef-2026.zip
[INFO] Using archive: /content/birdclef-2026-download/download/birdclef-2026.zip
[INFO] Archive size: 14.97 GB


In [21]:
# Step 4a: Inspect the archive so the visible payload is explicit before extraction.
print_banner('[INSPECT] Listing archive contents')

ARCHIVE_FILE_SIZES = build_archive_file_sizes(archive_path)
members = sorted(ARCHIVE_FILE_SIZES.keys())
print(f'[INFO] Archive member count: {len(members)}')
print(f'[INFO] Manifest entries with file sizes loaded: {len(ARCHIVE_FILE_SIZES)}')

if STRICT_MANIFEST_SIZE_VALIDATION:
    missing_manifest_entries = [entry for entry in EXPECTED_ENTRIES if not expected_sizes_for_entry(entry, ARCHIVE_FILE_SIZES)]
    if missing_manifest_entries:
        raise ValueError('Archive manifest missing expected entries: ' + ', '.join(missing_manifest_entries))
    print('[OK] Archive manifest contains all expected top-level entries.')

if SKIP_ARCHIVE_PREVIEW and FAST_MODE:
    print('[INFO] Archive preview skipped (FAST_MODE enabled).')
else:
    preview_count = min(30, len(members))
    print(f'[INFO] Showing first {preview_count} archive members:')
    for name in members[:preview_count]:
        print(f'  - {name}')




[INSPECT] Listing archive contents
[INFO] Archive member count: 46213
[INFO] Manifest entries with file sizes loaded: 46213
[OK] Archive manifest contains all expected top-level entries.
[INFO] Archive preview skipped (FAST_MODE enabled).


In [22]:
# Step 5: Preserve the original Kaggle archive in Google Drive before extraction.
print_banner('[STEP 5/8] Copying raw archive into Google Drive')

drive_archive_path = RAW_ARCHIVE_DIR / archive_path.name
if (
    SKIP_RAW_ARCHIVE_COPY_IF_EXISTS
    and drive_archive_path.exists()
    and drive_archive_path.stat().st_size == archive_path.stat().st_size
):
    print(f'[SKIP] Raw archive already present with same size: {drive_archive_path}')
else:
    copy_file_with_progress(archive_path, drive_archive_path)



[STEP 5/8] Copying raw archive into Google Drive
[SKIP] Raw archive already present with same size: /content/drive/MyDrive/birdclef-2026/raw/kaggle-download/birdclef-2026.zip


In [23]:
# Step 6: Extract the archive into temporary Colab runtime storage.
print_banner('[STEP 6/8] Extracting the Kaggle archive in runtime storage')

runtime_entries_exist = all((EXTRACT_DIR / entry).exists() for entry in EXPECTED_ENTRIES)
runtime_manifest_valid = runtime_entries_exist and not STRICT_MANIFEST_SIZE_VALIDATION
if runtime_entries_exist and STRICT_MANIFEST_SIZE_VALIDATION:
    runtime_manifest_valid = True
    for entry in EXPECTED_ENTRIES:
        ok, message = path_matches_manifest_size(entry, EXTRACT_DIR / entry, ARCHIVE_FILE_SIZES, show_progress=False)
        if not ok:
            runtime_manifest_valid = False
            print(f'[WARN] Runtime extract validation failed for {entry}: {message}')
            break
    if runtime_manifest_valid:
        print('[OK] Existing extracted runtime data matches archive manifest sizes.')

if SKIP_EXTRACT_IF_EXPECTED_EXISTS and runtime_entries_exist and runtime_manifest_valid:
    print('[SKIP] Reusing existing extracted runtime dataset; all expected entries are present.')
else:
    extract_zip_with_progress(archive_path, EXTRACT_DIR)
print(f'[INFO] Extracted directory summary: {summarize_directory(EXTRACT_DIR)}')



[STEP 6/8] Extracting the Kaggle archive in runtime storage
[OK] Existing extracted runtime data matches archive manifest sizes.
[SKIP] Reusing existing extracted runtime dataset; all expected entries are present.
[INFO] Extracted directory summary: 8 direct entries | deep stats skipped


In [24]:
# Step 7: Copy the extracted visible competition files into Google Drive.
# We keep an explicit placeholder for hidden test soundscapes so the folder layout
# matches the project plan without incorrectly implying that the hidden audio is present.
print_banner('[STEP 7/8] Syncing extracted dataset into Google Drive')

archive_refreshed_for_mismatch = False

def redownload_archive_and_refresh_extract(reason: str) -> None:
    global archive_path, ARCHIVE_FILE_SIZES
    print_banner('[RECOVERY] Redownloading archive due to size mismatch')
    print(f'[RECOVERY] Reason: {reason}')

    archive_candidate = DOWNLOAD_DIR / f'{COMPETITION}.zip'
    if archive_candidate.exists():
        print(f'[RECOVERY] Removing current runtime archive: {archive_candidate}')
        archive_candidate.unlink()

    cmd = [
        'kaggle', 'competitions', 'download',
        '-c', COMPETITION,
        '-p', str(DOWNLOAD_DIR),
    ]
    print('[RUN] ' + ' '.join(cmd))
    subprocess.run(cmd, check=True)

    archives = sorted(DOWNLOAD_DIR.glob('*.zip'))
    if not archives:
        raise FileNotFoundError('No competition archive was downloaded during recovery.')
    archive_path = archive_candidate if archive_candidate.exists() else archives[0]
    if not zipfile.is_zipfile(archive_path):
        raise RuntimeError(f'Redownloaded archive is not a valid zip: {archive_path}')

    ARCHIVE_FILE_SIZES = build_archive_file_sizes(archive_path)
    print(f'[OK] Reloaded archive manifest entries: {len(ARCHIVE_FILE_SIZES)}')
    extract_zip_with_progress(archive_path, EXTRACT_DIR)

for entry in EXPECTED_ENTRIES:
    src = EXTRACT_DIR / entry
    dst = DATA_DIR / entry
    print(f'[SYNC] Processing entry: {entry}')
    if not src.exists():
        print(f'[WARN] Expected entry not found in extracted archive: {entry}')
        continue

    if STRICT_MANIFEST_SIZE_VALIDATION:
        src_ok, src_message = path_matches_manifest_size(entry, src, ARCHIVE_FILE_SIZES, show_progress=False)
        if not src_ok:
            print(f'[WARN] Runtime source size validation failed for {entry}: {src_message}')
            if REDOWNLOAD_ARCHIVE_ON_SIZE_MISMATCH and not archive_refreshed_for_mismatch:
                redownload_archive_and_refresh_extract(f'Runtime source mismatch for {entry}: {src_message}')
                archive_refreshed_for_mismatch = True
                src = EXTRACT_DIR / entry
                src_ok, src_message = path_matches_manifest_size(entry, src, ARCHIVE_FILE_SIZES, show_progress=False)
            if not src_ok:
                raise RuntimeError(f'Runtime source still failed size validation for {entry}: {src_message}')

    if SKIP_SYNC_IF_ENTRY_EXISTS and dst.exists():
        if STRICT_MANIFEST_SIZE_VALIDATION:
            dst_ok, dst_message = path_matches_manifest_size(entry, dst, ARCHIVE_FILE_SIZES, show_progress=True)
            if dst_ok:
                print(f'[SKIP] Destination already matches archive manifest sizes: {entry}')
                continue

            print(f'[WARN] Destination size mismatch for {entry}: {dst_message}')
            if REDOWNLOAD_ARCHIVE_ON_SIZE_MISMATCH and not archive_refreshed_for_mismatch:
                redownload_archive_and_refresh_extract(f'Destination mismatch for {entry}: {dst_message}')
                archive_refreshed_for_mismatch = True
                src = EXTRACT_DIR / entry
            print(f'[SYNC] Recopying destination due to size mismatch: {dst}')
        else:
            if src.is_dir() and dst.is_dir():
                try:
                    has_any_content = any(dst.iterdir())
                except Exception:
                    has_any_content = False
                if has_any_content:
                    print(f'[SKIP] Destination already exists and is non-empty: {dst}')
                    continue
            if src.is_file() and dst.is_file():
                same_size = dst.stat().st_size == src.stat().st_size
                if (not SKIP_SYNC_IF_FILE_SAME_SIZE) or same_size:
                    print(f'[SKIP] Destination file already exists ({"same size" if same_size else "size check disabled"}): {dst}')
                    continue
                print(f'[SYNC] Destination exists but size differs; recopying: {dst}')

    if src.is_dir():
        if not (USE_RSYNC_FOR_DIRECTORY_SYNC and command_exists('rsync')):
            remove_existing_path(dst)
        copy_directory_with_progress(src, dst)
    else:
        remove_existing_path(dst)
        copy_file_with_progress(src, dst)

    if STRICT_MANIFEST_SIZE_VALIDATION:
        dst_ok, dst_message = path_matches_manifest_size(entry, dst, ARCHIVE_FILE_SIZES, show_progress=False)
        if not dst_ok:
            raise RuntimeError(f'Post-sync size validation failed for {entry}: {dst_message}')
        print(f'[OK] Post-sync size validation passed: {entry}')
    print(f'[SYNC] Completed entry: {entry}')

placeholder = DATA_DIR / 'test_soundscapes_placeholder'
placeholder.mkdir(parents=True, exist_ok=True)
note = placeholder / 'README.txt'
note.write_text(
    'Hidden test_soundscapes are not available in the initial competition download. '
    'Kaggle only exposes the hidden test data to scored submission notebooks.',
    encoding='utf-8'
)
print(f'[INFO] Placeholder written to: {note}')
print('[OK] Drive data sync complete.')



[STEP 7/8] Syncing extracted dataset into Google Drive
[SYNC] Processing entry: train_audio


Validating train_audio:   0%|          | 0/35549 [00:00<?, ?file/s]

[SKIP] Destination already matches archive manifest sizes: train_audio
[SYNC] Processing entry: train_soundscapes


Validating train_soundscapes:   0%|          | 0/10658 [00:00<?, ?file/s]

[SKIP] Destination already matches archive manifest sizes: train_soundscapes
[SYNC] Processing entry: train.csv
[SKIP] Destination already matches archive manifest sizes: train.csv
[SYNC] Processing entry: taxonomy.csv
[SKIP] Destination already matches archive manifest sizes: taxonomy.csv
[SYNC] Processing entry: sample_submission.csv
[SKIP] Destination already matches archive manifest sizes: sample_submission.csv
[SYNC] Processing entry: train_soundscapes_labels.csv
[SKIP] Destination already matches archive manifest sizes: train_soundscapes_labels.csv
[SYNC] Processing entry: recording_location.txt
[SKIP] Destination already matches archive manifest sizes: recording_location.txt
[INFO] Placeholder written to: /content/drive/MyDrive/birdclef-2026/data/test_soundscapes_placeholder/README.txt
[OK] Drive data sync complete.


In [25]:
# Step 8: Verify that the expected Drive artifacts exist and report their sizes.
print_banner('[STEP 8/8] Verifying Drive artifacts')

missing = [str(path) for path in REQUIRED_DRIVE_ARTIFACTS if not path.exists()]
if missing:
    raise FileNotFoundError('Missing expected Drive artifacts:\n' + '\n'.join(missing))

archive_size_gb = drive_archive_path.stat().st_size / (1024 ** 3)
print(f'[OK] Raw archive present: {drive_archive_path} ({archive_size_gb:.2f} GB)')

for path in REQUIRED_DRIVE_ARTIFACTS:
    if path.is_dir():
        deep_stats = VERIFY_DEEP_DIRECTORY_STATS and not FAST_MODE
        print(f'[OK] {path.name}: {summarize_directory(path, deep=deep_stats)}')
    else:
        print(f'[OK] {path.name}: {format_bytes(path.stat().st_size)}')

print('[OK] Verification complete.')



[STEP 8/8] Verifying Drive artifacts
[OK] Raw archive present: /content/drive/MyDrive/birdclef-2026/raw/kaggle-download/birdclef-2026.zip (14.97 GB)
[OK] train_audio: 206 direct entries | deep stats skipped
[OK] train_soundscapes: 10658 direct entries | deep stats skipped
[OK] train.csv: 6.47 MB
[OK] taxonomy.csv: 13.37 KB
[OK] sample_submission.csv: 16.32 KB
[OK] train_soundscapes_labels.csv: 131.47 KB
[OK] recording_location.txt: 204.00 B
[OK] Verification complete.


In [26]:
# Final report: Print the Drive manifest with a progress bar so long listings stay visible.
print_banner('[REPORT] Final Drive manifest')

def print_manifest_summary(root: Path) -> None:
    top_level = sorted(root.iterdir())
    print(f'[INFO] Top-level entry count: {len(top_level)}')
    for path in top_level:
        if path.is_dir():
            print(f'[DIR]  {path.name}: {summarize_directory(path)}')
        else:
            print(f'[FILE] {path.name}: {format_bytes(path.stat().st_size)}')

def walk_manifest(root: Path) -> None:
    paths = sorted(root.rglob('*'))
    print(f'[INFO] Full manifest item count: {len(paths)}')
    for path in tqdm(paths, desc='Printing full manifest', unit='item'):
        rel = path.relative_to(root)
        if path.is_dir():
            print(f'[DIR]  {rel}')
        else:
            print(f'[FILE] {rel} ({format_bytes(path.stat().st_size)})')

print_manifest_summary(DRIVE_ROOT)
if PRINT_FULL_MANIFEST:
    walk_manifest(DRIVE_ROOT)
else:
    print('[INFO] Full manifest printing is disabled by default for speed. Set PRINT_FULL_MANIFEST=True to enable.')



[REPORT] Final Drive manifest
[INFO] Top-level entry count: 5
[DIR]  data: 8 direct entries | deep stats skipped
[DIR]  docs: 0 direct entries | deep stats skipped
[DIR]  outputs: 3 direct entries | deep stats skipped
[DIR]  raw: 1 direct entries | deep stats skipped
[DIR]  secrets: 4 direct entries | deep stats skipped
[INFO] Full manifest printing is disabled by default for speed. Set PRINT_FULL_MANIFEST=True to enable.


## Next Step

After this notebook completes, use Drive as the persistent store and copy only the working subset you need into Colab runtime storage for training or analysis.
